## setup

In [1]:
# import stuff
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
# load data from folder
folder = Path('../data/processed')

# filtered datasets with mortgage rates
sold = pd.read_csv(folder / 'wk4_5_sold_clean.csv', low_memory = False)

In [3]:
# sold.head()
sold.shape

(448026, 50)

In [4]:
date_cols = ['CloseDate',
             'PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']

sold[date_cols] = sold[date_cols].apply(pd.to_datetime, errors = 'coerce')

## feature engineering

In [5]:
# ft eng
sold['price_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']

# normalizes price across sizes
sold['price_per_sqft'] = sold['ClosePrice'] / sold['LivingArea']

# enables time-series analysis
sold['year'] = sold['CloseDate'].dt.year
sold['month'] = sold['CloseDate'].dt.month
# df['YrMo'] = df['CloseDate'].dt.to_period('M')

# captures full price reduction history
sold['close_to_original_list_ratio'] = sold['ClosePrice'] / sold['OriginalListPrice']

# measures time from listing to accepted offer
sold['listing_to_contract_days'] = sold['PurchaseContractDate'] - sold['ListingContractDate']

# measures time from purchase date to close date
sold['contract_to_close_days'] = sold['CloseDate'] - sold['PurchaseContractDate']

# check that engineered columns were created
sold[['price_ratio',
      'price_per_sqft',
      'year',
      'month',
      'close_to_original_list_ratio',
      'listing_to_contract_days',
      'contract_to_close_days'
      ]].head()

,price_ratio,price_per_sqft,year,month,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days
0,0.480962,210.526316,2024.0,1.0,0.480962,777 days,65 days
1,1.072510,412.867275,2024.0,1.0,1.072510,114 days,919 days
2,1.094743,410.334347,2024.0,1.0,1.094743,255 days,778 days
3,1.000000,562.098501,NaN,NaN,1.000000,NaT,NaT
4,1.000000,928.571429,2024.0,1.0,1.000000,32 days,34 days


## add school districts

In [14]:
# add school districts
school_gdf = gpd.read_file('../data/school_districts.geojson')

In [16]:
# filter by unified school district
filtered_school_gdf = school_gdf[school_gdf['DistrictType'] == 'Unified']

# only get districtname col & geometry for merging
filtered_school_gdf = filtered_school_gdf[['DistrictName', 'geometry']]

# check that changes have been made
filtered_school_gdf.head()

,DistrictName,geometry
0,Alameda Unified,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,Albany City Unified,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,Berkeley Unified,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,Castro Valley Unified,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,Emery Unified,"POLYGON ((-13613999.038 4555592.769, -13614126..."


In [26]:
sold_gdf = gpd.GeoDataFrame(
    sold,
    geometry = gpd.points_from_xy(sold['Longitude'], sold['Latitude']),
    crs = 'EPSG:4326'
)

sold_gdf.head()

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,year_month,rate_30yr_fixed,price_ratio,price_per_sqft,year,month,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days,geometry
0,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,37.566106,-122.327954,1 Baldwin Avenue 411,...,2024-01,6.6425,0.480962,210.526316,2024.0,1.0,0.480962,777 days,65 days,POINT (-122.32795 37.56611)
1,NaN,False,False,759900.0,522107581,2024-01-05,815000.0,32.659315,-117.096922,2811 C Avenue,...,2024-01,6.6425,1.072510,412.867275,2024.0,1.0,1.072510,114 days,919 days,POINT (-117.09692 32.65932)
2,NaN,False,False,739900.0,510919001,2024-01-05,810000.0,32.659284,-117.097174,2812 C Avenue,...,2024-01,6.6425,1.094743,410.334347,2024.0,1.0,1.094743,255 days,778 days,POINT (-117.09717 32.65928)
3,NaN,True,False,2100000.0,1067652762,NaT,2100000.0,35.277199,-120.646786,2054 Fixlini Street,...,2024-01,6.6425,1.000000,562.098501,NaN,NaN,1.000000,NaT,NaT,POINT (-120.64679 35.2772)
4,Tile,True,False,1950000.0,1063453216,2024-01-22,1950000.0,39.419080,-123.814842,31451 Bay View Avenue,...,2024-01,6.6425,1.000000,928.571429,2024.0,1.0,1.000000,32 days,34 days,POINT (-123.81484 39.41908)


In [28]:
# standardize coordinate systems so they match
# otherwise you get a crs mismatch error
if sold_gdf.crs != school_gdf.crs:
    sold_gdf = sold_gdf.to_crs(school_gdf.crs)

In [29]:
# spatially merge datasets
merged = gpd.sjoin(
    sold_gdf,
    filtered_school_gdf,
    how = 'left',
    predicate = 'within'
)

merged[['Latitude', 'Longitude', 'DistrictName']].head()

,Latitude,Longitude,DistrictName
0,37.566106,-122.327954,NaN
1,32.659315,-117.096922,NaN
2,32.659284,-117.097174,NaN
3,35.277199,-120.646786,San Luis Coastal Unified
4,39.419080,-123.814842,Fort Bragg Unified


In [30]:
district_null_pct = merged['DistrictName'].isna().sum() / merged['DistrictName'].shape[0]

print('Percentage of missing values in DistrictName column:',
      round(district_null_pct, 4)
      )

Percentage of missing values in DistrictName column: 0.2513


## segment analysis

In [31]:
metrics = ['PropertySubType',
               'CountyOrParish',
               'MLSAreaMajor',
               'ListOfficeName',
               'BuyerOfficeName']

print('SOLD DATASET:')

for metric in metrics:
    print(f'Summary statistics for {metric}:')
    print(sold[metric].describe(), '\n')

SOLD DATASET:
Summary statistics for PropertySubType:
count                    447164
unique                       20
top       SingleFamilyResidence
freq                     335582
Name: PropertySubType, dtype: object 

Summary statistics for CountyOrParish:
count          448026
unique             62
top       Los Angeles
freq           111175
Name: CountyOrParish, dtype: object 

Summary statistics for MLSAreaMajor:
count                387638
unique                 1091
top       699 - Not Defined
freq                  46350
Name: MLSAreaMajor, dtype: object 

Summary statistics for ListOfficeName:
count      448026
unique      19160
top       Compass
freq        31728
Name: ListOfficeName, dtype: object 

Summary statistics for BuyerOfficeName:
count      440892
unique      21877
top       Compass
freq        29629
Name: BuyerOfficeName, dtype: object 

